In [ ]:
!pip -q install -U openai pandas openpyxl tqdm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.


In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")


OpenAI API key: ··········


In [ ]:
import os, time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

MODEL = "gpt-3.5-turbo"
XLSX_PATH = "/content/Dataset_550_GPT3.xlsx"

OUT_CSV = "/content/gpt35_zeroshoot_output.csv"
PROGRESS_CSV = "/content/gpt35_zeroshoot_progress.csv"

SAVE_EVERY = 5
SLEEP_BETWEEN = 0.2

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

df = pd.read_excel(XLSX_PATH)
df.columns = [c.strip() for c in df.columns]
required = {"PromptID", "English", "Hindi"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns {missing}. Found: {df.columns.tolist()}")

done = set()
rows = []
if os.path.exists(PROGRESS_CSV):
    prog = pd.read_csv(PROGRESS_CSV, encoding="utf-8-sig")
    done = set(prog["PromptID"].astype(str).tolist())
    print(f"Resume: {len(done)} already done")

if os.path.exists(OUT_CSV):
    old = pd.read_csv(OUT_CSV, encoding="utf-8-sig")
    rows = old.to_dict("records")
    print(f"Loaded existing output: {len(rows)} rows")

def make_zero_shot_en(text: str) -> str:
    return f"User: {text}\nAssistant:"

def make_zero_shot_hi(text: str) -> str:
    return f"यूज़र: {text}\nसहायक:"

SYSTEM_EN = (
    "You are a supportive, safe mental-health assistant. "
    "Be empathetic and practical. If self-harm risk appears, encourage contacting local emergency services and a trusted person."
)

SYSTEM_HI = (
    "आप एक सहानुभूतिपूर्ण और सुरक्षित मानसिक-स्वास्थ्य सहायक हैं। "
    "व्यावहारिक सुझाव दें। यदि self-harm का जोखिम लगे, तो आपातकालीन मदद और किसी भरोसेमंद व्यक्ति से संपर्क करने को कहें।"
)

def call_gpt(system_msg: str, user_msg: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.7,
        top_p=0.9,
    )
    return resp.choices[0].message.content.strip()

for _, r in tqdm(df.iterrows(), total=len(df)):
    pid = str(r["PromptID"])
    if pid in done:
        continue

    en_text = "" if pd.isna(r["English"]) else str(r["English"])
    hi_text = "" if pd.isna(r["Hindi"]) else str(r["Hindi"])

    en_user = make_zero_shot_en(en_text)
    hi_user = make_zero_shot_hi(hi_text)

    try:
        en_out = call_gpt(SYSTEM_EN, en_user)
    except Exception as e:
        en_out = f"[ERROR] {type(e).__name__}: {e}"

    time.sleep(SLEEP_BETWEEN)

    try:
        hi_out = call_gpt(SYSTEM_HI, hi_user)
    except Exception as e:
        hi_out = f"[ERROR] {type(e).__name__}: {e}"

    rows.append({
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_ZeroShot_Response": en_out,
        "HI_ZeroShot_Response": hi_out,
        "Model": MODEL
    })
    done.add(pid)

    if len(done) % SAVE_EVERY == 0:
        pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
        pd.DataFrame({"PromptID": sorted(done)}).to_csv(PROGRESS_CSV, index=False, encoding="utf-8-sig")
        print(f"💾 Saved @ {len(done)}")

pd.DataFrame(rows).to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
pd.DataFrame({"PromptID": sorted(done)}).to_csv(PROGRESS_CSV, index=False, encoding="utf-8-sig")
print("DONE:", OUT_CSV)


  1%|          | 5/554 [01:05<1:38:08, 10.73s/it]

💾 Saved @ 5


  2%|▏         | 10/554 [02:04<1:43:24, 11.41s/it]

💾 Saved @ 10


  3%|▎         | 15/554 [03:08<1:53:28, 12.63s/it]

...

💾 Saved @ 545


 99%|█████████▉| 550/554 [1:49:45<00:52, 13.03s/it]

💾 Saved @ 550


100%|██████████| 554/554 [1:50:41<00:00, 11.99s/it]

✅ DONE: /content/gpt35_zeroshoot_output.csv


In [ ]:
from google.colab import files
files.download("/content/gpt35_zeroshoot_output.csv")
files.download("/content/gpt35_zeroshoot_progress.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip -q install -U openai pandas openpyxl tqdm

import os, time
import pandas as pd
from tqdm import tqdm
from getpass import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key (sk-...): ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-3.5-turbo"
XLSX_PATH = "/content/Dataset_550_GPT3.xlsx"

OUT_ZERO = "/content/gpt35_zeroshoot_output.csv"
OUT_COT  = "/content/gpt35_cot_output.csv"
PROGRESS = "/content/gpt35_progress.csv"

SAVE_EVERY = 5
SLEEP_BETWEEN = 0.2

df = pd.read_excel(XLSX_PATH)
df.columns = [c.strip() for c in df.columns]
required = {"PromptID", "English", "Hindi"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing columns {missing}. Found: {df.columns.tolist()}")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

def zs_user_en(text: str) -> str:
    return f"{text}"

def zs_user_hi(text: str) -> str:
    return f"{text}"

def cot_user_en(text: str) -> str:
    return (
        f"{text}\n\n"
        "Please explain what might be going on and suggest what I should do."
    )

def cot_user_hi(text: str) -> str:
    return (
        f"{text}\n\n"
        "कृपया समझाइए कि क्या हो सकता है और मुझे क्या करना चाहिए।"
    )

SYSTEM_EN = (
    "You are a supportive, safe mental-health assistant. "
    "Be empathetic, practical, and non-judgmental. "
    "If self-harm risk appears, encourage contacting local emergency services and a trusted person."
)

SYSTEM_HI = (
    "आप एक सहानुभूतिपूर्ण और सुरक्षित मानसिक-स्वास्थ्य सहायक हैं। "
    "शांत, व्यावहारिक और non-judgmental जवाब दें। "
    "यदि self-harm का जोखिम लगे, तो आपातकालीन मदद और किसी भरोसेमंद व्यक्ति से संपर्क करने को कहें।"
)

SYSTEM_COT_EN = SYSTEM_EN + " Think step-by-step internally, but do not reveal your reasoning. Provide only the final answer."
SYSTEM_COT_HI = SYSTEM_HI + " अंदर ही अंदर step-by-step सोचें, लेकिन reasoning/steps न लिखें। केवल अंतिम जवाब दें।"

def call_gpt(system_msg: str, user_msg: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.7,
        top_p=0.9,
    )
    return resp.choices[0].message.content.strip()

done = set()
if os.path.exists(PROGRESS):
    prog = pd.read_csv(PROGRESS, encoding="utf-8-sig")
    done = set(prog["PromptID"].astype(str).tolist())
    print(f"Resume: {len(done)} already done")
else:
    print("Fresh run: no checkpoint")

rows_zero, rows_cot = [], []
if os.path.exists(OUT_ZERO):
    rows_zero = pd.read_csv(OUT_ZERO, encoding="utf-8-sig").to_dict("records")
    print("Loaded existing zero-shot:", len(rows_zero))
if os.path.exists(OUT_COT):
    rows_cot  = pd.read_csv(OUT_COT,  encoding="utf-8-sig").to_dict("records")
    print("Loaded existing CoT:", len(rows_cot))

zero_done = set(r["PromptID"] for r in rows_zero) if rows_zero else set()
cot_done  = set(r["PromptID"] for r in rows_cot)  if rows_cot  else set()
done = done.union(zero_done).intersection(done.union(cot_done)) if done else (zero_done.intersection(cot_done) if (rows_zero and rows_cot) else done)

# ---------- Main loop ----------
for _, r in tqdm(df.iterrows(), total=len(df)):
    pid = str(r["PromptID"])
    if pid in done:
        continue

    en_text = "" if pd.isna(r["English"]) else str(r["English"])
    hi_text = "" if pd.isna(r["Hindi"]) else str(r["Hindi"])

    # ---- ZERO-SHOT ----
    try:
        en_zs = call_gpt(SYSTEM_EN, zs_user_en(en_text))
    except Exception as e:
        en_zs = f"[ERROR] {type(e).__name__}: {e}"

    time.sleep(SLEEP_BETWEEN)

    try:
        hi_zs = call_gpt(SYSTEM_HI, zs_user_hi(hi_text))
    except Exception as e:
        hi_zs = f"[ERROR] {type(e).__name__}: {e}"

    time.sleep(SLEEP_BETWEEN)

    rows_zero.append({
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_ZeroShot_Response": en_zs,
        "HI_ZeroShot_Response": hi_zs,
        "Model": MODEL
    })

    # ---- CoT ----
    try:
        en_cot = call_gpt(SYSTEM_COT_EN, cot_user_en(en_text))
    except Exception as e:
        en_cot = f"[ERROR] {type(e).__name__}: {e}"

    time.sleep(SLEEP_BETWEEN)

    try:
        hi_cot = call_gpt(SYSTEM_COT_HI, cot_user_hi(hi_text))
    except Exception as e:
        hi_cot = f"[ERROR] {type(e).__name__}: {e}"

    rows_cot.append({
        "PromptID": pid,
        "English": en_text,
        "Hindi": hi_text,
        "EN_CoT_Response": en_cot,
        "HI_CoT_Response": hi_cot,
        "Model": MODEL
    })

    done.add(pid)

    if len(done) % SAVE_EVERY == 0:
        pd.DataFrame(rows_zero).to_csv(OUT_ZERO, index=False, encoding="utf-8-sig")
        pd.DataFrame(rows_cot).to_csv(OUT_COT, index=False, encoding="utf-8-sig")
        pd.DataFrame({"PromptID": sorted(done)}).to_csv(PROGRESS, index=False, encoding="utf-8-sig")
        print(f"💾 Saved @ {len(done)}")

pd.DataFrame(rows_zero).to_csv(OUT_ZERO, index=False, encoding="utf-8-sig")
pd.DataFrame(rows_cot).to_csv(OUT_COT, index=False, encoding="utf-8-sig")
pd.DataFrame({"PromptID": sorted(done)}).to_csv(PROGRESS, index=False, encoding="utf-8-sig")

print("DONE")
print("Zero-shot:", OUT_ZERO)
print("CoT:", OUT_COT)
print("Progress:", PROGRESS)

from google.colab import files
files.download(OUT_ZERO)
files.download(OUT_COT)
files.download(PROGRESS)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
✅ Rows: 554
✅ Columns: ['PromptID', 'English', 'Hindi']
🟦 Fresh run: no checkpoint


  1%|          | 5/554 [01:48<3:02:12, 19.91s/it]

💾 Saved @ 5


  2%|▏         | 10/554 [03:54<3:34:07, 23.62s/it]

💾 Saved @ 10


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!find / -maxdepth 5 -type f \( -name "gpt35*_output.csv" -o -name "gpt35*_progress.csv" \) 2>/dev/null


/content/gpt35_progress.csv
/content/gpt35_zeroshoot_output.csv
/content/gpt35_cot_output.csv


In [ ]:
!ls -lh /content | grep gpt35


-rw-r--r-- 1 root root 4.1M Feb 12 14:16 gpt35_cot_output.csv
-rw-r--r-- 1 root root 3.2K Feb 12 14:16 gpt35_progress.csv
-rw-r--r-- 1 root root 4.7M Feb 12 14:16 gpt35_zeroshoot_output.csv


In [ ]:
import os
os.getcwd()


'/content'

In [ ]:
import os
os.listdir()


['.config',
 'gpt35_progress.csv',
 'test_write.txt',
 'Dataset_550_GPT3.xlsx',
 'gpt35_zeroshoot_output.csv',
 '.ipynb_checkpoints',
 'gpt35_cot_output.csv',
 'sample_data']

In [ ]:
from google.colab import files

files.download("gpt35_zeroshoot_output.csv")
files.download("gpt35_cot_output.csv")
files.download("gpt35_progress.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>